In [ ]:
import json
import logging
import os
import time
from pathlib import Path

import torch
import torch.nn as nn
import torchaudio

from torch.utils.data import DataLoader, Subset
from collections import defaultdict

from sklearn.model_selection import train_test_split

from src.tokenizer import *
from src.asr_metrics import *
from src.asr_model import *
from src.dataset import *
from src.lexicon import *

In [ ]:
torch.backends.cudnn.benchmark = False
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
use_bfloat = True

log = logging.getLogger(__name__)
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)

In [ ]:
# SpecAugment

class SpecAugment(nn.Module):
    def __init__(self, time_mask_param=50, freq_mask_param=27, num_masks=2):
        super().__init__()
        self.time_aug = torchaudio.transforms.TimeMasking(time_mask_param)
        self.freq_aug = torchaudio.transforms.FrequencyMasking(freq_mask_param)
        self.num_masks = num_masks

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, T, H) → transpose to (B, H, T) for torchaudio
        x = x.transpose(1, 2)
        for _ in range(self.num_masks):
            x = self.time_aug(x)
            x = self.freq_aug(x)
        return x.transpose(1, 2)

In [ ]:
train_path = os.path.join('data', 'metadata_normal.tsv')
val_path = os.path.join('data', 'metadata_val_normal.tsv')
audio_dir = os.path.join('data', 'audio')
cache = os.path.join('data', 'cache')
save_dir = os.path.join('checkpoints', 'asr')
token_dir = os.path.join('data', 'tokenizer.json')
lexicon_dir = os.path.join('data', 'lexicon.txt')
dict_dir = os.path.join('data', 'dictionary.json')
ssl_ckpt = os.path.join('checkpoints', 'ssl', 'checkpoint_0055000.pt')

In [ ]:
def save_checkpoint(model, optimizer, step, loss, save_dir, tag=""):
    if tag != "":
        path = Path(save_dir) / f"checkpoint_{step:07d}{tag}.pt"
    else:
        path = Path(save_dir) / f"checkpoint_{step:07d}.pt"
    torch.save({
        "step":                step,
        "loss":                loss,
        "model":    model.state_dict(),
        "optimizer": optimizer.state_dict(),
    }, path)
    log.info(f"Saved checkpoint -> {path}")
    return path

def load_checkpoint(model, optimizer, path, device):
    ckpt = torch.load(path, map_location=device)
    model.load_state_dict(ckpt["model"])
    optimizer.load_state_dict(ckpt["optimizer"])
    log.info(f"Resumed from step {ckpt['step']}, loss={ckpt['loss']:.4f}")
    return ckpt["step"]

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
log.info(f"Device: {device}")

lr = 1e-3
weight_decay = 1e-2
batch_size = 12

total_updates = 60_000

if not os.path.exists(token_dir):
    log.info("Tokenizer not found, building from metadata")
    
    tokenizer = GraphemeTokenizer()
    tokenizer.build_vocab_from_metadata(train_path)
    tokenizer.save(token_dir)
else:
    tokenizer = GraphemeTokenizer.load(token_dir)
    
if not os.path.exists(lexicon_dir):
    log.info(f"Lexicon not found, building from dictionary at {dict_dir}")
    build_lexicon_from_dict(
        dict_dir,
        tokenizer,
        lexicon_dir
    )
else:
    log.info("Lexicon loaded.")
    

def collate_fn_with_tokenizer(batch):
    return collate_fn_asr(batch, tokenizer)

full_dataset = NepaliSpeechDataset(audio_base_dir=audio_dir, metadata_path=train_path, use_memory_map=True, cache_dir=cache, for_ssl=False)
n = len(full_dataset)

lengths = [
    len(list(glib.graphemes(str(full_dataset.metadata.iloc[i]['transcript']))))
    for i in range(n)
]
buckets = pd.cut(lengths, bins=[0,10,20,30,50,999], labels=[0,1,2,3,4])

train_idx, val_idx = train_test_split(
    range(n), test_size=0.05, random_state=42, stratify=buckets
)

train_dataset = Subset(full_dataset, train_idx)
val_dataset   = Subset(full_dataset, val_idx)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    pin_memory=True,
    collate_fn=collate_fn_with_tokenizer
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    pin_memory=True,
    collate_fn=collate_fn_with_tokenizer
)

model = load_asr_model(
    ssl_ckpt,
    vocab_size=tokenizer.get_vocab_size(),
    device=device,
    freeze_encoder=False
)

spec_aug = SpecAugment(
    time_mask_param=50,
    freq_mask_param=27
).to(device)

base_lrs = [1e-3, 1e-3, 1e-4, 1e-5]  # norm, head, transformer, CNN

optimizer = torch.optim.AdamW([
    {"params": model.pre_head_norm.parameters(), "lr": base_lrs[0]},
    {"params": model.ctc_head.parameters(),      "lr": base_lrs[1]},
    {"params": model.context.parameters(),       "lr": base_lrs[2]},
    {"params": model.encoder.parameters(),       "lr": base_lrs[3]},
], weight_decay=1e-2, betas=(0.9, 0.98))

scaler = torch.GradScaler(device=device, enabled=use_bfloat)

step = 0
metrics_history = defaultdict(list)

In [ ]:
# base_lrs = [1e-3, 1e-3, 5e-5, 5e-6]

# # Don't restore optimizer LRs from checkpoint — set manually
# # apply_lr will handle the correct scaled values from step 12000 onwards

step = load_checkpoint(
    model,
    optimizer,
    'checkpoints/asr/checkpoint_0058000_best.pt',
    device
)

# optimizer = torch.optim.AdamW([
#     {"params": model.pre_head_norm.parameters(), "lr": base_lrs[0]},
#     {"params": model.ctc_head.parameters(),      "lr": base_lrs[1]},
#     {"params": model.context.parameters(),       "lr": base_lrs[2]},
#     {"params": model.encoder.parameters(),       "lr": base_lrs[3]},
# ], weight_decay=1e-2, betas=(0.9, 0.98))

In [ ]:
def train_step(model, batch, optimizer, scaler, spec_aug=None, device="cuda"):
    waveform       = batch["waveforms"].to(device, non_blocking=True)
    audio_lengths  = batch["audio_lengths"].to(device, non_blocking=True)
    targets        = batch["targets"].to(device, non_blocking=True)
    target_lengths = batch["target_lengths"].to(device, non_blocking=True)

    dtype = torch.bfloat16 if use_bfloat else torch.float32

    effective_spec_aug = spec_aug if any(
        p.requires_grad for p in model.encoder.parameters()
    ) else None

    with torch.autocast(device_type="cuda", dtype=dtype, enabled=use_bfloat):
        log_probs, frame_lengths = model(waveform, audio_lengths, effective_spec_aug)

        ctc = F.ctc_loss(
            log_probs.transpose(0, 1),
            targets, frame_lengths, target_lengths,
            blank=0, reduction="mean", zero_infinity=True,
        )
        ctc_val = ctc.item()

        log_probs_f32 = log_probs.float()
        entropy     = -(log_probs_f32.exp() * log_probs_f32).sum(-1).mean()
        entropy_val = entropy.item()
        blank_frac  = (log_probs.argmax(-1) == 0).float().mean().item()

        loss = ctc - 0.10 * entropy

    if torch.isnan(loss) or torch.isinf(loss):
        log.warning(f"NaN/Inf loss | ctc={ctc_val:.3f} entropy={entropy_val:.3f}")
        optimizer.zero_grad()
        return None

    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    scaler.step(optimizer)
    scaler.update()
    optimizer.zero_grad()

    return loss.item(), ctc_val, entropy_val, blank_frac

In [ ]:
def get_lr_scale(step, warmup_steps, decay_at=30000, decay_factor=0.3):
    if step < warmup_steps:
        return step / warmup_steps
    elif step < decay_at:
        return 1.0
    else:
        return decay_factor

def apply_lr(optimizer, base_lrs, scale):
    for g, base_lr in zip(optimizer.param_groups, base_lrs):
        g["lr"] = base_lr * scale

In [ ]:
def train_asr(total_steps, model, optimizer, train_loader, val_loader,
              tokenizer, scaler, spec_aug, device, metrics_history,
              base_lrs, warmup_steps=2000, cnn_unfreeze_at=5000, initial_step=0,
              lr_decay_at=30000, lr_decay_factor=0.3):

    save_every        = 5000
    log_interval      = 100
    evaluate_interval = 1000
    cnn_unfrozen      = False
    space_id          = tokenizer.char2id['<space>']

    if initial_step >= cnn_unfreeze_at:
        model.unfreeze_cnn()
        cnn_unfrozen = True
        log.info("CNN already unfrozen (resume)")

    val_loss, cer, wer = float("inf"), float("inf"), float("inf")
    best_cer    = float("inf")
    loader_iter = iter(train_loader)
    model.train()
    optimizer.zero_grad()
    t0 = time.time()

    pbar = tqdm(range(initial_step, total_steps), desc="Training", dynamic_ncols=True)
    for step in pbar:

        # Flat LR with single decay step (no cosine, no norm intervention)
        scale = get_lr_scale(step, warmup_steps, lr_decay_at, lr_decay_factor)
        apply_lr(optimizer, base_lrs, scale)

        if not cnn_unfrozen and step >= cnn_unfreeze_at:
            model.unfreeze_cnn()
            cnn_unfrozen = True
            log.info(f"CNN unfrozen at step {step}")

        try:
            batch = next(loader_iter)
        except StopIteration:
            loader_iter = iter(train_loader)
            batch = next(loader_iter)

        result = train_step(model, batch, optimizer, scaler,
                            spec_aug=spec_aug, device=device)
        if result is None:       # ← was: if loss is None (loss not yet defined)
            continue

        loss, ctc_val, entropy_val, blank_frac = result

        if step % log_interval == 0:
            elapsed = time.time() - t0
            ups     = log_interval / elapsed
            log.info(
                f"step={step:>6d} | loss={loss:.4f} | ctc={ctc_val:.4f} | "
                f"entropy={entropy_val:.4f} | blank={blank_frac:.3f} | "
                f"lr_head={optimizer.param_groups[1]['lr']:.2e} | "
                f"lr_enc={optimizer.param_groups[2]['lr']:.2e} | "
                f"{ups:.1f} up/s"
            )
            t0 = time.time()

        if step % 2000 == 0:
            geom = compute_geometry(model, val_loader, device)
            log.info(
                f"[Geometry] norm_mean={geom['norm_mean']:.3f} | "
                f"norm_std={geom['norm_std']:.3f} | "
                f"active_dims={geom['active_dims']} | "
                f"isotropy={geom['isotropy']:.3f}"
            )
            # No norm intervention — just log and move on

        if step % evaluate_interval == 0:
            val_loss, cer, wer = evaluate(model, val_loader, tokenizer, device)

            # Greedy space fraction on one val batch — cheap generalization signal
            model.eval()
            with torch.no_grad():
                batch_diag = next(iter(val_loader))
                lp, fl = model(
                    batch_diag["waveforms"].to(device),
                    batch_diag["audio_lengths"].to(device)
                )
                argmax = lp.argmax(-1)
                total, spaces = 0, 0
                for i in range(len(fl)):
                    seq = argmax[i, :fl[i]].tolist()
                    nb  = [t for t in seq if t != 0]
                    total  += len(nb)
                    spaces += nb.count(space_id)
            space_frac = spaces / max(total, 1)
            model.train()

            log.info(
                f"step={step:>7d} | val_loss={val_loss:.4f} | "
                f"cer={cer*100:.2f}% | wer={wer*100:.2f}% | "
                f"space_frac={space_frac:.4f}"
            )
            metrics_history["step"].append(step)
            metrics_history["loss_val"].append(val_loss)
            metrics_history["cer_val"].append(cer)
            metrics_history["wer_val"].append(wer)
            metrics_history["space_frac"].append(space_frac)

            if cer < best_cer:
                best_cer = cer
                save_checkpoint(model, optimizer, step, val_loss, save_dir, tag="_best")

        if step % save_every == 0 and step > initial_step:
            save_checkpoint(model, optimizer, step, val_loss, save_dir)

    pbar.close()
    log.info("Training complete.")
    save_checkpoint(model, optimizer, step, val_loss, save_dir, tag="_end")

    with open(Path(save_dir) / "metrics.json", "w") as f:
        json.dump(metrics_history, f, indent=2)

In [ ]:
# train_asr(
#     total_steps=total_updates,
#     model=model,
#     optimizer=optimizer,
#     train_loader=train_loader,
#     val_loader=val_loader,
#     tokenizer=tokenizer,
#     scaler=scaler,
#     spec_aug=spec_aug,
#     device=device,
#     metrics_history=defaultdict(list),
#     base_lrs=base_lrs,
#     warmup_steps=2000,
#     cnn_unfreeze_at=5000,
#     initial_step=step,
# )

In [ ]:
model.eval()
with torch.no_grad():
    batch = next(iter(val_loader))
    log_probs, frame_lengths = model(
        batch["waveforms"].to(device),
        batch["audio_lengths"].to(device)
    )
    probs = log_probs.exp()
    
    space_id = tokenizer.char2id['<space>']
    blank_id = 0
    
    # Average probability per token across all frames
    avg_probs = probs.mean(dim=(0, 1))  # (vocab_size,)
    
    print(f"Avg blank prob:  {avg_probs[blank_id]:.4f}")
    print(f"Avg space prob:  {avg_probs[space_id]:.4f}")
    print(f"Non-blank, non-space: {1 - avg_probs[blank_id] - avg_probs[space_id]:.4f}")
    
    # Frame-level: what fraction of frames have space as TOP prediction
    top_pred = probs.argmax(-1)  # (B, T)
    print(f"\nFrames where space is top pred: {(top_pred == space_id).float().mean():.4f}")
    print(f"Frames where blank is top pred: {(top_pred == blank_id).float().mean():.4f}")
    
    # What does a raw token sequence look like before collapsing?
    for i in range(3):
        seq = top_pred[i, :frame_lengths[i]].tolist()
        # Show run-length encoded version
        from itertools import groupby
        rle = [(k, sum(1 for _ in g)) for k, g in groupby(seq)]
        tokens = [(tokenizer.id2char.get(k, '?'), n) for k, n in rle]
        print(f"\nSample {i} RLE ({len(seq)} frames → {len(rle)} runs):")
        print(tokens[:30])
        print("greedy:", tokenizer.decode_ctc(seq))
        print("ref:   ", batch["transcripts"][i])

In [ ]:
model.eval()
with torch.no_grad():
    batch = next(iter(val_loader))
    log_probs, frame_lengths = model(
        batch["waveforms"].to(device),
        batch["audio_lengths"].to(device)
    )
decoder = build_beam_decoder(lexicon_dir, tokenizer)

hypotheses = beam_decode_batch(log_probs, frame_lengths, decoder, tokenizer)

for i in range(min(10, len(hypotheses))):
    print(f"[{i}]")
    print(f"  beam:  {hypotheses[i]}")
    print(f"  greedy: {tokenizer.decode_ctc(log_probs[i].argmax(-1).tolist())}")
    print(f"  ref:    {batch['transcripts'][i]}")
    print()

In [ ]:
model.eval()
batch = next(iter(val_loader))
waveform = batch["waveforms"].to(device)
lengths  = batch["audio_lengths"].to(device)

with torch.no_grad():
    log_probs, frame_lengths = model(waveform, lengths)

for i in range(3):
    ids  = log_probs[i, :frame_lengths[i]].argmax(dim=-1).tolist()
    print("raw argmax:", ids[:30])
    print("decoded:   ", tokenizer.decode_ctc(ids))
    print("reference: ", batch["transcripts"][i])

In [ ]:
def apply_space_boost(log_probs, space_id, space_scale=4.0):
    logits = log_probs.clone()
    logits[:, :, space_id] += math.log(space_scale)
    return F.log_softmax(logits, dim=-1)

decoder = build_beam_decoder(lexicon_dir, tokenizer)

space_id = tokenizer.char2id[tokenizer.SPACE_TOKEN]

model.eval()
batch = next(iter(train_loader))
waveform = batch["waveforms"].to(device)
lengths  = batch["audio_lengths"].to(device)

with torch.no_grad():
    log_probs, frame_lengths = model(waveform, lengths)

for scale in [2.0, 4.0, 8.0, 16.0, 32.0]:
    boosted = apply_space_boost(log_probs, space_id, scale)
    
    # Check space fraction
    argmax = boosted.argmax(dim=-1)
    sf = (argmax == space_id).float().mean().item()
    print(f"\n{'='*60}")
    print(f"scale={scale:.0f} | space_frac={sf:.3f} (target: 0.10-0.13)")
    print(f"{'='*60}")
    
    hypotheses = beam_decode_batch(boosted, frame_lengths, decoder)
    for i in range(3):
        print("decoded:   ", hypotheses[i])
        print("reference: ", batch["transcripts"][i])
        print()